In [0]:
# group by syntax

# select cols, agg from tbl
# group by cols
# order by agg

# syntax: dataframe.groupBy(col).agg(agg-sum,min,max,count,avg)
# col as col_name == sql
# col.alias(colname) == pyspark

In [0]:
# Scenario:
# You want to calculate the average salary and total bonus per department.

# GroupBy and Aggregation Example
aggregated_df = df.groupBy("department") \
                 .agg(avg("salary").alias("avg_salary"), sum("bonus").alias("total_bonus"))
aggregated_df=aggregated_df.withColumn("total_bonus",coalesce(col("total_bonus"),lit(0))) # lit stands for literal
aggregated_df.display()


In [0]:
# #pandas

# merge ==== we should be clear both dataframes should have same column name 
# join ===== we will join based on indexes

In [0]:
# Scenario:
# Assume you have another dataset with department managers. 
# Join this data with the original dataset to include manager names.

# Create a second DataFrame (department managers)
managers_data = [( "SME", "John"), ("Testing", "Mary"), ("Engineering", "Steve"), ("Junior Developer", "Kate"),("Developer","Sravya"),("IT","Venkat")]
managers_columns = ["department", "manager_name"]
managers_df = spark.createDataFrame(managers_data, managers_columns)


In [0]:
# joined_df=df.join(managers_df,df.department==managers_df.department,how="inner").drop(df.department)
joined_df=df.join(managers_df,on="department",how="inner")

In [0]:
# Scenario:
# You want to replace missing bonus values with 0.
cleaned_df = df.fillna({"bonus": 0})

In [0]:
# %sql
# select *,
# case when salary>1l then high 
# when salary>60k and <1l then med
# else low end as salary_category
# -- from cleaned_df

In [0]:
# syntax of case when in pyspark

# df=df.withColumn("colname",
#                  when (col(colname)>val,"res")
#                  .when(col(colname)<val,"res")
#                  .otherwise("res"))


In [0]:
# Scenario:
# You want to add a new column salary_category based on the salary values.
updated_df = df.withColumn("salary_category", 
               when(col("salary") >= 100000, "High")
              .when(col("salary").between(60000, 100000), "Medium")
              .otherwise("Low"))

updated_df.select("id", "name", "salary", "salary_category").display()


In [0]:
def mask_name(name):
    return name[0:2] + "*" * (len(name) - 4)+name[-2::1] # J***
# udf stands for user defined functions
mask_name_udf = udf(mask_name, StringType()) # udf(funname,fun_rturn_datatype)

# Apply the UDF to create a new column
masked_df = df.withColumn("masked_name", mask_name_udf(col("name")))
masked_df.select("id", "name", "masked_name").display()


In [0]:
def years_to_retirement(age):
    return 65 - age

# Register the UDF
retirement_udf = udf(years_to_retirement, IntegerType())
df = df.withColumn("years_to_retirement", retirement_udf(col("age")))

In [0]:
# Scenario:
# You want to add a column categorizing cities into regions (e.g., "East", "West", "Central").


def city_to_region(city):
    if city in ["New York", "Houston"]:
        return "East"
    elif city in ["Los Angeles", "San Francisco"]:
        return "West"
    elif city == "Chicago":
        return "Central"
    else:
        return "Unknown"

# Register the UDF
region_udf = udf(city_to_region, StringType())

# Apply the UDF to create a new column
region_df = df.withColumn("region", region_udf(col("city")))

# Display the result
region_df.select("id", "city", "region").display()

